> **EXPERIMENTO LEGADO — NÃO É EVIDÊNCIA PUBLICÁVEL.** `DeepNPTS aproximado` não é uma implementação canônica do DeepNPTS e os resultados deste caderno não pertencem à avaliação mensal corrigida. O método foi renomeado para `VizinhosHistoricos` no pipeline atual. Para resultados, protocolo e comparação estatística reproduzíveis, use `04_avaliacao_mensal_corrigida.ipynb` e `resultados/avaliacao_mensal_corrigida/`.


# Resultados dos Modelos Avançados
## DilatedRNN, DeepAR experimental e DeepNPTS aproximado

Este notebook resume a rodada complementar de modelos avançados. 

Os arquivos lidos vêm de:

```text
resultados/experimentos_redes_avancadas/
```

Modelos avaliados:

| Modelo | Papel no experimento |
|---|---|
| `DilatedRNN` | rede recorrente multiescala com subamostragens dilatadas das features temporais |
| `DeepAR_exp` | aproximação experimental do DeepAR com LSTM probabilística e perda Gaussiana |
| `DeepNPTS_aprox` | baseline não paramétrico inspirado no NPTS, com ponderação por similaridade e recência |

A comparação usa as mesmas localidades, frequências, divisão temporal e métricas do pipeline principal. O critério principal continua sendo o maior `R2_wm2`.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'cadernos_jupyter' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTADOS_EXP = PROJECT_ROOT / 'resultados' / 'experimentos_redes_avancadas'

status = pd.read_csv(RESULTADOS_EXP / 'status_execucao.csv')
metricas = pd.read_csv(RESULTADOS_EXP / 'metricas_experimentos_todas.csv')
comparacao_diaria = pd.read_csv(RESULTADOS_EXP / 'diaria' / 'comparacao_com_modelos_oficiais.csv')
comparacao_mensal = pd.read_csv(RESULTADOS_EXP / 'mensal' / 'comparacao_com_modelos_oficiais.csv')

colunas_resultado = [
    'Frequencia', 'Localidade', 'Pais', 'Modelo',
    'MAE_wm2', 'RMSE_wm2', 'R2_wm2', 'nRMSE_percentual_wm2',
]
colunas_comparacao = [
    'Frequencia', 'Localidade', 'Pais', 'Modelo', 'Origem',
    'MAE_wm2', 'RMSE_wm2', 'R2_wm2', 'nRMSE_percentual_wm2',
]
colunas_metricas = ['MAE_wm2', 'RMSE_wm2', 'R2_wm2', 'nRMSE_percentual_wm2']

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

def arredondar(df, casas=4):
    return df.round({
        'MAE_wm2': 2,
        'RMSE_wm2': 2,
        'R2_wm2': casas,
        'nRMSE_percentual_wm2': 2,
    })

def melhores_por_localidade(df):
    melhores = df.loc[df.groupby('Localidade')['R2_wm2'].idxmax()].copy()
    return melhores.sort_values(['Localidade']).reset_index(drop=True)

frequencia_ordem = {'diaria': 0, 'mensal': 1}


## 1. Status da execução

Antes de analisar desempenho, a primeira verificação é confirmar se todos os treinos terminaram corretamente. Nesta rodada foram avaliadas 10 localidades, 2 frequências e 3 modelos experimentais, totalizando 60 execuções.

In [2]:
status_resumo = (
    status
    .groupby(['Frequencia', 'Status'])
    .size()
    .rename('Quantidade')
    .reset_index()
)

status_resumo

,Frequencia,Status,Quantidade
0,diaria,ok,30
1,mensal,ok,30


## 2. Média dos modelos experimentais

Esta tabela resume o desempenho médio de cada candidato experimental. Ela serve para observar o comportamento geral dos modelos antes de olhar localidade por localidade.

In [3]:
resumo_experimentos = (
    metricas
    .groupby(['Frequencia', 'Modelo'])[colunas_metricas]
    .mean()
    .reset_index()
)
resumo_experimentos['ordem_frequencia'] = resumo_experimentos['Frequencia'].map(frequencia_ordem)
resumo_experimentos = (
    resumo_experimentos
    .sort_values(['ordem_frequencia', 'RMSE_wm2'])
    .drop(columns='ordem_frequencia')
    .reset_index(drop=True)
)

arredondar(resumo_experimentos)

,Frequencia,Modelo,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,diaria,DilatedRNN,37.70,49.71,0.6565,26.86
1,diaria,DeepNPTS_aprox,39.32,51.65,0.6286,27.86
2,diaria,DeepAR_exp,41.44,54.40,0.5858,29.33
3,mensal,DeepNPTS_aprox,12.96,16.39,0.9157,8.10
4,mensal,DilatedRNN,15.25,18.73,0.8778,9.29
5,mensal,DeepAR_exp,61.02,68.57,0.0096,34.21


## 3. Resultados por localidade

As próximas tabelas mostram os resultados de todos os modelos experimentais em cada localidade. Essa visão é importante porque o desempenho médio pode esconder diferenças regionais.

In [4]:
resultados_diarios = (
    metricas.loc[metricas['Frequencia'].eq('diaria'), colunas_resultado]
    .sort_values(['Localidade', 'R2_wm2'], ascending=[True, False])
    .reset_index(drop=True)
)
resultados_mensais = (
    metricas.loc[metricas['Frequencia'].eq('mensal'), colunas_resultado]
    .sort_values(['Localidade', 'R2_wm2'], ascending=[True, False])
    .reset_index(drop=True)
)

print('Resultados experimentais - frequência diária')
display(arredondar(resultados_diarios))

print('Resultados experimentais - frequência mensal')
display(arredondar(resultados_mensais))

Resultados experimentais - frequência diária


,Frequencia,Localidade,Pais,Modelo,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,diaria,BMW San Luis Potosi,Mexico,DilatedRNN,33.69,45.26,0.6027,19.26
1,diaria,BMW San Luis Potosi,Mexico,DeepAR_exp,34.87,48.26,0.5483,20.54
2,diaria,BMW San Luis Potosi,Mexico,DeepNPTS_aprox,35.83,49.66,0.5217,21.13
3,diaria,BYD Camacari,Brasil,DilatedRNN,39.15,50.00,0.3811,20.48
4,diaria,BYD Camacari,Brasil,DeepNPTS_aprox,40.55,51.70,0.3381,21.18
5,diaria,BYD Camacari,Brasil,DeepAR_exp,42.15,52.29,0.3231,21.42
6,diaria,Ford Rouge Electric Vehicle Center,EUA,DilatedRNN,45.19,57.91,0.6586,39.89
7,diaria,Ford Rouge Electric Vehicle Center,EUA,DeepNPTS_aprox,45.78,60.58,0.6264,41.73
8,diaria,Ford Rouge Electric Vehicle Center,EUA,DeepAR_exp,46.89,61.07,0.6203,42.07
9,diaria,GM Factory Zero,EUA,DilatedRNN,45.98,59.93,0.6381,41.34


Resultados experimentais - frequência mensal


,Frequencia,Localidade,Pais,Modelo,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,mensal,BMW San Luis Potosi,Mexico,DeepNPTS_aprox,18.65,23.00,0.7382,9.28
1,mensal,BMW San Luis Potosi,Mexico,DilatedRNN,20.60,24.36,0.7064,9.83
2,mensal,BMW San Luis Potosi,Mexico,DeepAR_exp,39.64,44.58,0.0167,17.99
3,mensal,BYD Camacari,Brasil,DeepNPTS_aprox,13.85,17.64,0.7779,7.47
4,mensal,BYD Camacari,Brasil,DilatedRNN,16.51,22.32,0.6445,9.45
5,mensal,BYD Camacari,Brasil,DeepAR_exp,33.28,37.39,0.0025,15.83
6,mensal,Ford Rouge Electric Vehicle Center,EUA,DeepNPTS_aprox,13.21,16.30,0.9558,10.16
7,mensal,Ford Rouge Electric Vehicle Center,EUA,DilatedRNN,13.82,17.01,0.9518,10.60
8,mensal,Ford Rouge Electric Vehicle Center,EUA,DeepAR_exp,69.06,77.50,-0.0005,48.30
9,mensal,GM Factory Zero,EUA,DilatedRNN,14.61,17.22,0.9502,10.74


## 4. Melhor modelo experimental por localidade

Aqui entram somente os três candidatos experimentais. Para cada localidade, o notebook seleciona o maior `R2_wm2`.

In [5]:
melhores_exp_diaria = melhores_por_localidade(metricas[metricas['Frequencia'].eq('diaria')])[colunas_resultado]
melhores_exp_mensal = melhores_por_localidade(metricas[metricas['Frequencia'].eq('mensal')])[colunas_resultado]

print('Melhor experimental por localidade - diária')
display(arredondar(melhores_exp_diaria))

print('Melhor experimental por localidade - mensal')
display(arredondar(melhores_exp_mensal))

Melhor experimental por localidade - diária


,Frequencia,Localidade,Pais,Modelo,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,diaria,BMW San Luis Potosi,Mexico,DilatedRNN,33.69,45.26,0.6027,19.26
1,diaria,BYD Camacari,Brasil,DilatedRNN,39.15,50.00,0.3811,20.48
2,diaria,Ford Rouge Electric Vehicle Center,EUA,DilatedRNN,45.19,57.91,0.6586,39.89
3,diaria,GM Factory Zero,EUA,DilatedRNN,45.98,59.93,0.6381,41.34
4,diaria,Hyundai Metaplant Georgia,EUA,DilatedRNN,45.77,57.08,0.5214,30.59
5,diaria,Lucid AMP 1 Casa Grande,EUA,DilatedRNN,23.67,36.11,0.8204,15.54
6,diaria,Rivian Normal,EUA,DilatedRNN,46.60,60.01,0.6338,36.42
7,diaria,Tesla Fremont Factory,EUA,DilatedRNN,25.47,36.59,0.8737,17.89
8,diaria,Tesla Gigafactory Nevada,EUA,DilatedRNN,29.00,39.36,0.8589,19.18
9,diaria,Tesla Gigafactory Texas,EUA,DilatedRNN,42.44,54.82,0.5763,27.97


Melhor experimental por localidade - mensal


,Frequencia,Localidade,Pais,Modelo,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,mensal,BMW San Luis Potosi,Mexico,DeepNPTS_aprox,18.65,23.00,0.7382,9.28
1,mensal,BYD Camacari,Brasil,DeepNPTS_aprox,13.85,17.64,0.7779,7.47
2,mensal,Ford Rouge Electric Vehicle Center,EUA,DeepNPTS_aprox,13.21,16.30,0.9558,10.16
3,mensal,GM Factory Zero,EUA,DilatedRNN,14.61,17.22,0.9502,10.74
4,mensal,Hyundai Metaplant Georgia,EUA,DeepNPTS_aprox,11.25,13.76,0.9370,6.94
5,mensal,Lucid AMP 1 Casa Grande,EUA,DilatedRNN,9.52,10.58,0.9798,4.29
6,mensal,Rivian Normal,EUA,DeepNPTS_aprox,18.41,21.94,0.9184,12.19
7,mensal,Tesla Fremont Factory,EUA,DilatedRNN,11.64,13.39,0.9799,6.07
8,mensal,Tesla Gigafactory Nevada,EUA,DilatedRNN,9.57,12.09,0.9834,5.44
9,mensal,Tesla Gigafactory Texas,EUA,DeepNPTS_aprox,9.50,11.84,0.9579,5.67


## 5. Comparação com os modelos oficiais

Nesta etapa, os três candidatos experimentais são comparados com XGBoost, MLP, RNN e LSTM. O objetivo é verificar se algum modelo avançado supera o melhor resultado oficial em cada localidade.

In [6]:
melhores_diaria = melhores_por_localidade(comparacao_diaria)[colunas_comparacao]
melhores_mensal = melhores_por_localidade(comparacao_mensal)[colunas_comparacao]

print('Melhor modelo geral por localidade - diária')
display(arredondar(melhores_diaria))

print('Melhor modelo geral por localidade - mensal')
display(arredondar(melhores_mensal))

Melhor modelo geral por localidade - diária


,Frequencia,Localidade,Pais,Modelo,Origem,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,diaria,BMW San Luis Potosi,Mexico,DilatedRNN,experimento,33.69,45.26,0.6027,19.26
1,diaria,BYD Camacari,Brasil,MLP,oficial,38.75,49.67,0.3892,20.35
2,diaria,Ford Rouge Electric Vehicle Center,EUA,DilatedRNN,experimento,45.19,57.91,0.6586,39.89
3,diaria,GM Factory Zero,EUA,RNN,oficial,45.97,59.51,0.6432,41.04
4,diaria,Hyundai Metaplant Georgia,EUA,RNN,oficial,45.87,56.93,0.5240,30.50
5,diaria,Lucid AMP 1 Casa Grande,EUA,DilatedRNN,experimento,23.67,36.11,0.8204,15.54
6,diaria,Rivian Normal,EUA,MLP,oficial,46.85,59.50,0.6401,36.11
7,diaria,Tesla Fremont Factory,EUA,DilatedRNN,experimento,25.47,36.59,0.8737,17.89
8,diaria,Tesla Gigafactory Nevada,EUA,XGBoost,oficial,28.47,39.26,0.8596,19.13
9,diaria,Tesla Gigafactory Texas,EUA,DilatedRNN,experimento,42.44,54.82,0.5763,27.97


Melhor modelo geral por localidade - mensal


,Frequencia,Localidade,Pais,Modelo,Origem,MAE_wm2,RMSE_wm2,R2_wm2,nRMSE_percentual_wm2
0,mensal,BMW San Luis Potosi,Mexico,DeepNPTS_aprox,experimento,18.65,23.00,0.7382,9.28
1,mensal,BYD Camacari,Brasil,DeepNPTS_aprox,experimento,13.85,17.64,0.7779,7.47
2,mensal,Ford Rouge Electric Vehicle Center,EUA,RNN,oficial,12.10,15.38,0.9606,9.59
3,mensal,GM Factory Zero,EUA,RNN,oficial,13.04,16.12,0.9563,10.06
4,mensal,Hyundai Metaplant Georgia,EUA,DeepNPTS_aprox,experimento,11.25,13.76,0.9370,6.94
5,mensal,Lucid AMP 1 Casa Grande,EUA,XGBoost,oficial,5.61,6.65,0.9920,2.70
6,mensal,Rivian Normal,EUA,DeepNPTS_aprox,experimento,18.41,21.94,0.9184,12.19
7,mensal,Tesla Fremont Factory,EUA,RNN,oficial,10.13,12.17,0.9834,5.52
8,mensal,Tesla Gigafactory Nevada,EUA,DilatedRNN,experimento,9.57,12.09,0.9834,5.44
9,mensal,Tesla Gigafactory Texas,EUA,DeepNPTS_aprox,experimento,9.50,11.84,0.9579,5.67


## 6. Contagem de vitórias

A contagem de vitórias mostra quantas localidades foram vencidas por modelos oficiais e quantas foram vencidas pelos experimentais.

In [7]:
vencedores = pd.concat(
    [
        melhores_diaria.assign(Frequencia='diaria'),
        melhores_mensal.assign(Frequencia='mensal'),
    ],
    ignore_index=True,
)

contagem_vitorias = (
    vencedores
    .groupby(['Frequencia', 'Origem', 'Modelo'])
    .size()
    .rename('Vitorias')
    .reset_index()
    .sort_values(['Frequencia', 'Origem', 'Vitorias'], ascending=[True, True, False])
)

display(contagem_vitorias)

resumo_experimentos_feedback = (
    metricas
    .groupby(['Frequencia', 'Modelo'])[colunas_metricas]
    .mean()
    .reset_index()
)

def feedback_frequencia(frequencia, comparacao):
    melhores = melhores_por_localidade(comparacao)
    contagem = melhores.groupby(['Origem', 'Modelo']).size().sort_values(ascending=False)
    melhor_medio = (
        resumo_experimentos_feedback[resumo_experimentos_feedback['Frequencia'].eq(frequencia)]
        .sort_values('R2_wm2', ascending=False)
        .iloc[0]
    )
    melhor_geral = melhores.loc[melhores['R2_wm2'].idxmax()]
    linhas = [
        f"- **{origem} / {modelo}**: {qtd} vitória(s)"
        for (origem, modelo), qtd in contagem.items()
    ]
    return f"""
### {frequencia.capitalize()}

**Vitórias na comparação com modelos oficiais:**
{chr(10).join(linhas)}

**Experimental com melhor média de R²:** {melhor_medio['Modelo']} com R² médio **{melhor_medio['R2_wm2']:.4f}** e RMSE médio **{melhor_medio['RMSE_wm2']:.2f} W/m²**.

**Melhor caso geral:** {melhor_geral['Localidade']} com **{melhor_geral['Modelo']}** ({melhor_geral['Origem']}), R² = **{melhor_geral['R2_wm2']:.4f}**.
"""

texto_feedback = "\n".join(
    [
        feedback_frequencia('diaria', comparacao_diaria),
        feedback_frequencia('mensal', comparacao_mensal),
        """
### Feedback técnico

Os modelos avançados melhoraram algumas localidades, mas ainda devem ser tratados como rodada complementar. A `DilatedRNN` se mostrou competitiva no fluxo diário. No fluxo mensal, o `DeepNPTS_aprox` teve bom desempenho relativo, enquanto o `DeepAR_exp` ficou sensível à configuração experimental usada.

Para usar esses modelos como resultado principal, seria recomendável ampliar a validação temporal, revisar hiperparâmetros e documentar melhor as aproximações em relação às versões canônicas de DeepAR e NPTS.
""",
    ]
)

display(Markdown(texto_feedback))


,Frequencia,Origem,Modelo,Vitorias
0,diaria,experimento,DilatedRNN,5
1,diaria,oficial,MLP,2
2,diaria,oficial,RNN,2
3,diaria,oficial,XGBoost,1
4,mensal,experimento,DeepNPTS_aprox,5
5,mensal,experimento,DilatedRNN,1
6,mensal,oficial,RNN,3
7,mensal,oficial,XGBoost,1


## 7. Leitura dos resultados

A contagem acima resume a comparação entre os candidatos experimentais e os modelos oficiais. O ponto principal é que os experimentos avançados são úteis para investigação, mas não substituem automaticamente o pipeline oficial.

Feedbacks de leitura:

- quando um modelo experimental vence em uma localidade, isso indica potencial de ganho local;
- quando o modelo oficial vence, o pipeline principal continua sendo a opção mais estável;
- resultados médios ajudam a enxergar tendência geral, mas a decisão final deve considerar localidade por localidade;
- o `DeepAR_exp` é uma aproximação simplificada e precisa de validação metodológica adicional antes de ser tratado como implementação principal.


## 8. Reexecução opcional

A célula abaixo fica comentada para evitar sobrescrever os CSVs ao abrir o notebook. Ela deve ser usada apenas se for necessário treinar novamente os modelos avançados.

In [8]:
# from experimentos_redes_avancadas import rodar_experimentos, selecionar_localidades
#
# metricas_completas = rodar_experimentos(
#     frequencias=['diaria', 'mensal'],
#     localidades=selecionar_localidades(None),
# )
# metricas_completas